In [ ]:
# Paths (relative to the repository root, so the notebooks run on any machine)
# Expected layout:  <repo>/notebooks/ (this file), <repo>/data/, <repo>/models/,
#                   <repo>/figures/, <repo>/results/, <repo>/feature_importance/
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = ROOT / "data"
MODELS_DIR = ROOT / "models"
FIG_DIR = ROOT / "figures"
RESULTS_DIR = ROOT / "results"
FI_DIR = ROOT / "feature_importance"
for _d in (DATA_DIR, MODELS_DIR, FIG_DIR, RESULTS_DIR, FI_DIR):
    _d.mkdir(parents=True, exist_ok=True)


In [1]:
import pandas as pd
import numpy as np

from rdkit import Chem
from rdkit.Chem import Descriptors

from mordred import Calculator, descriptors

import warnings
warnings.filterwarnings("ignore")

In [2]:
df = pd.read_csv(str(DATA_DIR / "MetalCytoToxDB.csv"))

In [3]:
df["SMILES_Ligands"].head()

0    N#Cc1ccc(C(c2ccc(C#N)cc2)n2cncn2)cc1.N#Cc1ccc(...
1    COc1cc(/C=C/C(=O)/C=C([O-])/C=C/c2ccc(O)c(OC)c...
2    COc1cc(/C=C/C(=O)/C=C([O-])/C=C/c2ccc(O)c(OC)c...
3    COc1cc(/C=C/C(=O)/C=C([O-])/C=C/c2ccc(O)c(OC)c...
4    COc1cc(/C=C/C(=O)/C=C([O-])/C=C/c2ccc(O)c(OC)c...
Name: SMILES_Ligands, dtype: str

In [4]:
df["Molecule"] = df["SMILES_Ligands"].apply(Chem.MolFromSmiles)

In [5]:
invalid = df["Molecule"].isnull().sum()

print("Invalid SMILES:", invalid)

Invalid SMILES: 0


In [6]:
print("Valid Molecules:", len(df))

Valid Molecules: 26801


In [7]:
descriptor_names = [
    "MolWt",
    "MolLogP",
    "TPSA",
    "NumHDonors",
    "NumHAcceptors",
    "NumRotatableBonds",
]

In [8]:
#Step 8: Descriptor Calculation Function
def calculate_descriptors(mol):
    return {
        "MolWt": Descriptors.MolWt(mol),
        "MolLogP": Descriptors.MolLogP(mol),
        "TPSA": Descriptors.TPSA(mol),
        "NumHDonors": Descriptors.NumHDonors(mol),
        "NumHAcceptors": Descriptors.NumHAcceptors(mol),
        "NumRotatableBonds": Descriptors.NumRotatableBonds(mol),
    }

In [9]:
rdkit_desc = df["Molecule"].apply(calculate_descriptors)

rdkit_desc = pd.DataFrame(rdkit_desc.tolist())

In [10]:
rdkit_desc.head()

,MolWt,MolLogP,TPSA,NumHDonors,NumHAcceptors,NumRotatableBonds
0,868.365,5.76712,156.58,0,8,9
1,537.052,2.77732,99.05,2,6,8
2,537.052,2.77732,99.05,2,6,8
3,537.052,2.77732,99.05,2,6,8
4,537.052,2.77732,99.05,2,6,8


In [11]:
df_final = pd.concat(
    [
        df.reset_index(drop=True),
        rdkit_desc.reset_index(drop=True)
    ],
    axis=1
)

In [12]:
print(df_final.shape)

(26801, 31)


In [13]:
from rdkit.Chem import Descriptors

descriptor_names = [name for name, _ in Descriptors._descList]

print("Number of RDKit descriptors:", len(descriptor_names))

Number of RDKit descriptors: 217


In [14]:
from rdkit.Chem import Descriptors

def calc_all_descriptors(mol):
    values = []

    for name, function in Descriptors._descList:
        try:
            values.append(function(mol))
        except:
            values.append(np.nan)

    return values

In [15]:
rdkit_all = df["Molecule"].apply(calc_all_descriptors)

In [16]:
rdkit_all = pd.DataFrame(
    rdkit_all.tolist(),
    columns=descriptor_names
)

In [17]:
rdkit_all.head()

,MaxAbsEStateIndex,MaxEStateIndex,MinAbsEStateIndex,MinEStateIndex,qed,SPS,MolWt,HeavyAtomMolWt,ExactMolWt,NumValenceElectrons,...,fr_sulfide,fr_sulfonamd,fr_sulfone,fr_term_acetylene,fr_tetrazole,fr_thiazole,fr_thiocyan,fr_thiophene,fr_unbrch_alkane,fr_urea
0,8.917602,8.917602,0.0,-0.445988,0.165816,10.109375,868.365,831.069,867.263429,308,...,0,0,0,0,0,0,0,0,0,0
1,11.873843,11.873843,0.0,-0.472242,0.260867,11.157895,537.052,503.788,536.197664,202,...,0,0,0,0,0,0,0,0,0,0
2,11.873843,11.873843,0.0,-0.472242,0.260867,11.157895,537.052,503.788,536.197664,202,...,0,0,0,0,0,0,0,0,0,0
3,11.873843,11.873843,0.0,-0.472242,0.260867,11.157895,537.052,503.788,536.197664,202,...,0,0,0,0,0,0,0,0,0,0
4,11.873843,11.873843,0.0,-0.472242,0.260867,11.157895,537.052,503.788,536.197664,202,...,0,0,0,0,0,0,0,0,0,0


In [18]:
df_ml = pd.concat(
    [
        df.reset_index(drop=True),
        rdkit_all.reset_index(drop=True)
    ],
    axis=1
)

In [19]:
print(df_ml.shape)

(26801, 242)


In [20]:
rdkit_all.to_csv(str(DATA_DIR / "RDKit_Descriptors.csv"), index=False)